# RFI explorer

Data | data - DPSS smooth model | kept-only residual (flags/v2), for one
antenna over one window, straight from `eigsep_data`: the metadata index
picks the rows, `load_bundle` joins the raw counts to their `flags@v2` and
`smooth_model@v0` companions on a common time/frequency axis.

No widgets: cell 2 prints the filespans available, cell 3 plots the one
you name in `SPAN` (and `MINUTES`, `ANTENNA`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from eigsep_data import MetadataIndex

#CAMPAIGN = "/mnt/data02/eigsep/marjum-2026-07"
CAMPAIGN = "/home/aparsons/projects/global21cm/eigsep/marjum-2026-07"
BAND_MHZ = (35.0, 250.0)
#PRODUCTS = ["flags@v2", "smooth_model@v0"]
PRODUCTS = []

idx = MetadataIndex(f"{CAMPAIGN}/data")
sel = idx.select()
print(sel.summary())
%matplotlib widget

## Filespans available

Contiguous stretches of observing, split wherever the data stops for more
than `GAP_S`. Times are `time_best` (header time where the clock was sane,
filename estimate otherwise) -- not the filename close times.

In [ ]:
GAP_S = 600

meta = sel.meta.assign(span=sel.visits(gap_s=GAP_S))
spans = pd.DataFrame(
    [
        dict(
            span=sid,
            start=pd.to_datetime(g.time_best.min(), unit="s"),
            end=pd.to_datetime(g.time_best.max(), unit="s"),
            hours=(g.time_best.max() - g.time_best.min()) / 3600,
            files=g.file.nunique(),
            rows=len(g),
            first_file=g.file.iloc[0],
            last_file=g.file.iloc[-1],
        )
        for sid, g in meta.groupby("span")
    ]
).set_index("span")

with pd.option_context("display.width", 200, "display.max_colwidth", 30):
    print(spans)

## Load and plot

Pick a span, how many minutes from its start to read, and an antenna.
Any window works, not just a whole span -- `idx.select(time=(lo, hi))`
takes real times, and `idx.select(files=(first, last))` takes an
inclusive filename range if that is what you want instead.

In [ ]:
SPAN = 13
MINUTES = 20
ANTENNA = "cross"   # or "box-gnd"

t0 = spans.loc[SPAN, "start"].timestamp()
t1 = spans.loc[SPAN, "end"].timestamp()
#window = idx.select(time=(t0, t0 + MINUTES * 60))
window = idx.select(time=(t1 - MINUTES * 60, t1))
#b = window.load_bundle(antenna=ANTENNA, products=PRODUCTS, band_mhz=BAND_MHZ)
bundles = {k: window.load_bundle(antenna=k, band_mhz=BAND_MHZ) for k in ('box-gnd', 'box-air', 'cross')}
b = bundles['cross']
inttime = b.meta['integration_time'][0]
bandwidth = 1e6 * (b.freqs_mhz[1] - b.freqs_mhz[0])
nsamples = 2 * inttime * bandwidth
auto_geometric_mean = np.sqrt(bundles['box-gnd'].data) * np.sqrt(bundles['box-air'].data)
zscore0 = np.abs(bundles['cross'].data) / auto_geometric_mean.clip(1, np.inf) * np.sqrt(nsamples)
mask_v0 = np.where(zscore0 > 20, 0, 1)
m = np.sum(mask_v0, axis=0)
#m_f = np.where(m > 0.95 * mask_v0.shape[0], 1, 0)
m_f = np.where(m > 0.9 * mask_v0.shape[0], 1, 0)
mask_v1 = mask_v0 * m_f[None, :]
m = np.sum(mask_v1, axis=1)
m_t = np.where(m == m_f.sum(), 1, 0)
mask_v2 = m_f[None,:] * m_t[:, None]

In [ ]:
plt.figure()
plt.imshow(mask_v2, interpolation=None)

In [ ]:
from hera_filters.dspec import dpss_operator, fit_solution_matrix

def dpss_filter(y, amat, fmat):
    '''Apply the provided DPSS filter matrices to data.'''
    model = amat @ (fmat @ y)
    return model

dat = bundles['box-air'].data
A_f = dpss_operator(b.freqs_mhz * 1e6, filter_centers=[0], filter_half_widths=[50e-9], eigenval_cutoff=[1e-12])[0]
F_f = fit_solution_matrix(m_f, A_f)
mdl = dpss_filter(dat.T, A_f, F_f).T.real
mdl *= m_t[:, None]
#A_t = dpss_operator(b.t, filter_centers=[0], filter_half_widths=[1/600], eigenval_cutoff=[1e-12])[0]
A_t = dpss_operator(b.t, filter_centers=[0], filter_half_widths=[1e-3], eigenval_cutoff=[1e-12])[0]
F_t = fit_solution_matrix(m_t, A_t)
mdl = dpss_filter(mdl, A_t, F_t).real
res = dat / mdl - 1
zscore = np.sqrt(nsamples) * res

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(8, 8), sharex=True, sharey=True)
minutes = (b.t - b.t.min()) / 60
extent = [b.freqs_mhz.min(), b.freqs_mhz.max(), minutes.max(), minutes.min()]
im = axes[0].imshow(bundles['box-air'].data, aspect="auto", extent=extent, norm=mcolors.LogNorm(vmin=1e3, vmax=1e8), cmap='plasma', interpolation=None)
im = axes[1].imshow(mdl, aspect="auto", extent=extent, norm=mcolors.LogNorm(vmin=1e3, vmax=1e8), cmap='plasma', interpolation=None)
im = axes[2].imshow(np.sqrt(nsamples) * np.abs(bundles['box-air'].data/mdl - 1), aspect="auto", extent=extent, norm=mcolors.LogNorm(vmin=1e0, vmax=1e2), cmap='plasma', interpolation=None)
#im = ax.imshow(np.abs(coh), aspect="auto", extent=extent, norm=mcolors.LogNorm(vmin=1e3, vmax=1e8), cmap='plasma', interpolation=None)
axes[0].set_xlabel("Frequency (MHz)")
fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
t_start = spans.loc[SPAN, "start"].strftime("%Y-%m-%d %H:%M:%SZ")
axes[0].set_ylabel(f"minutes from {t_start}")
fig.tight_layout()
plt.show()